In [2]:
import pandas as pd

In [7]:
df = pd.read_csv('../dataset.csv')

In [9]:
df.head()

,movie_id,title,year,directors,actors,genres
0,1,toy story,1995,John Lasseter,Tom Hanks Tim Allen Don Rickles Jim Varney Wal...,Animation Adventure Comedy Family Fantasy
1,2,goldeneye,1995,Martin Campbell,Pierce Brosnan Sean Bean Izabella Scorupco Fam...,Action Adventure Thriller
2,3,four rooms,1995,Allison Anders Alexandre Rockwell Robert Rodri...,Sammi Davis Amanda De Cadenet Valeria Golino M...,Comedy
3,4,get shorty,1995,Barry Sonnenfeld,John Travolta Gene Hackman Rene Russo Danny De...,Comedy Crime Thriller
4,5,copycat,1995,Jon Amiel,Sigourney Weaver Holly Hunter Dermot Mulroney ...,Drama Mystery Thriller


In [12]:
df.shape

(1681, 6)

In [17]:
df.duplicated().sum()

np.int64(0)

In [20]:
df.isna().sum()

movie_id       0
title          0
year           0
directors    126
actors       126
genres       120
dtype: int64

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1681 entries, 0 to 1680
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   movie_id   1681 non-null   int64
 1   title      1681 non-null   str  
 2   year       1681 non-null   int64
 3   directors  1555 non-null   str  
 4   actors     1555 non-null   str  
 5   genres     1561 non-null   str  
dtypes: int64(2), str(4)
memory usage: 78.9 KB


In [25]:
df = df.dropna()

In [31]:
df.shape

(1546, 6)

In [32]:
df['soup'] = (
    df['genres'].fillna('') + ' ' +
    df['directors'].fillna('') + ' ' +
    df['actors'].fillna('')
)

In [39]:
df.head()

,movie_id,title,year,directors,actors,genres,soup
0,1,toy story,1995,John Lasseter,Tom Hanks Tim Allen Don Rickles Jim Varney Wal...,Animation Adventure Comedy Family Fantasy,Animation Adventure Comedy Family Fantasy John...
1,2,goldeneye,1995,Martin Campbell,Pierce Brosnan Sean Bean Izabella Scorupco Fam...,Action Adventure Thriller,Action Adventure Thriller Martin Campbell Pier...
2,3,four rooms,1995,Allison Anders Alexandre Rockwell Robert Rodri...,Sammi Davis Amanda De Cadenet Valeria Golino M...,Comedy,Comedy Allison Anders Alexandre Rockwell Rober...
3,4,get shorty,1995,Barry Sonnenfeld,John Travolta Gene Hackman Rene Russo Danny De...,Comedy Crime Thriller,Comedy Crime Thriller Barry Sonnenfeld John Tr...
4,5,copycat,1995,Jon Amiel,Sigourney Weaver Holly Hunter Dermot Mulroney ...,Drama Mystery Thriller,Drama Mystery Thriller Jon Amiel Sigourney Wea...


In [42]:
df['soup'][0]

"animation adventure comedy family fantasy john lasseter tom hanks tim allen don rickles jim varney wallace shawn john ratzenberger annie potts john morris erik von detten laurie metcalf r. lee ermey sarah freeman penn jillette jack angel spencer aste greg berg lisa bradley kendall cunningham debi derryberry cody dorkin bill farmer craig good gregory grudt danielle judovits sam lasseter brittany levenbrown sherry lynn scott mcafee mickie mcgowan ryan o'donohue jeff pidgeon patrick pinney phil proctor jan rabson joe ranft andrew stanton shane sweet anthony burch carole jeghers john lasseter billy west"

In [41]:
df['soup'] = df['soup'].str.lower()
df['soup'] = df['soup'].str.replace('|', ' ', regex=False)
df['soup'] = df['soup'].str.replace(',', ' ', regex=False)

In [43]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer
cv = CountVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 1)
)

count_matrix = cv.fit_transform(df['soup'])

In [44]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity
cosine_sim = cosine_similarity(count_matrix, count_matrix)

In [46]:
cosine_sim

array([[1.        , 0.11834527, 0.03917304, ..., 0.15446945, 0.04850713,
        0.        ],
       [0.11834527, 1.        , 0.06567587, ..., 0.12948849, 0.09759001,
        0.01959655],
       [0.03917304, 0.06567587, 1.        , ..., 0.17144607, 0.16151457,
        0.        ],
       ...,
       [0.15446945, 0.12948849, 0.17144607, ..., 1.        , 0.10614898,
        0.02131523],
       [0.04850713, 0.09759001, 0.16151457, ..., 0.10614898, 1.        ,
        0.        ],
       [0.        , 0.01959655, 0.        , ..., 0.02131523, 0.        ,
        1.        ]], shape=(1546, 1546))

In [47]:
# Reset index for easy lookup
df = df.reset_index(drop=True)

def recommend(title, cosine_sim=cosine_sim, df=df, n=10):
    """
    Recommend movies similar to a given title.
    
    Parameters:
    - title: Movie title (partial match)
    - n: Number of recommendations
    """
    # Find movie
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    
    if len(matches) == 0:
        return f"❌ Movie '{title}' not found"
    
    idx = matches.index[0]
    
    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    
    # Get indices
    movie_indices = [i[0] for i in sim_scores]
    
    # Return recommendations
    return df[['title', 'genres', 'year']].iloc[movie_indices]

print("✅ Recommender function ready!")

✅ Recommender function ready!


In [48]:
# Test with different movies
test_movies = [
    "Toy Story",
    "Inception",
    "The Godfather",
    "Titanic",
    "The Matrix"
]

for movie in test_movies:
    print("=" * 60)
    print(f"🎬 Recommendations for: {movie}")
    print("=" * 60)
    try:
        print(recommend(movie, n=5))
    except Exception as e:
        print(f"Error: {e}")
    print()

🎬 Recommendations for: Toy Story
                              title  \
611                         fog the   
523                      pocahontas   
945                        hercules   
612            escape from new york   
420  american werewolf in london an   

                                                genres  year  
611                                    Horror Thriller  1980  
523   Animation Adventure Drama Family Musical Romance  1995  
945  Animation Adventure Comedy Family Fantasy Musi...  1997  
612                            Action Adventure Sci-Fi  1981  
420                                      Comedy Horror  1981  

🎬 Recommendations for: Inception
❌ Movie 'Inception' not found

🎬 Recommendations for: The Godfather
❌ Movie 'The Godfather' not found

🎬 Recommendations for: Titanic
                    title                           genres  year
20             braveheart      Biography Drama History War  1995
175    return of the jedi  Action Adventure Fantasy Sci-

In [50]:
import joblib

# Save artifacts
artifacts = {
    'cosine_sim': cosine_sim,
    'df': df,
    'count_matrix': count_matrix,
    'cv': cv
}

joblib.dump(artifacts, 'movie_recommender.pkl')
print("✅ Model saved as 'movie_recommender.pkl'")

✅ Model saved as 'movie_recommender.pkl'
